# iFood Case Técnico – Notebook 1: Processamento de Dados com PySpark

**Objetivo:** Carregar, limpar e unificar os três datasets (offers, profile, transactions) em um dataset enriquecido de interações cliente×oferta, pronto para modelagem.

**Premissas:**
- `age = 118` é um valor sentinela utilizado para clientes sem idade informada.
- O campo de ID da oferta varia por tipo de evento: `"offer id"` (com espaço) em *received* e *viewed*; `"offer_id"` (com underscore) em *completed*. O parsing extrai o valor correto em cada caso.
- Para ofertas do tipo *informational*, o sucesso é definido como *viewed* (não há evento de completion).
- Para fins de correspondência oferta→cliente, utilizamos a abordagem agregada: se qualquer reception foi seguida de qualquer completion (sem controle estrito de janela temporal), o par é marcado como convertido.
- O campo `time_since_test_start` está em **horas** (apesar do nome sugerir dias).

## 1. Configuração do Ambiente

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.sql.window import Window
import pandas as pd
import os
import json

# Para Databricks: o SparkSession já estará disponível como `spark`
spark = (
    SparkSession.builder
    .appName("iFood-DataProcessing")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
# Caminhos dos dados
# Para Databricks: RAW = "/dbfs/FileStore/ifood/raw"
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
RAW = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED, exist_ok=True)
print(f"RAW:       {RAW}")
print(f"PROCESSED: {PROCESSED}")

## 2. Carregamento dos Dados

Os arquivos JSON são arrays (`[{...}, ...]`). Utilizamos `pandas` para leitura e depois convertemos para Spark DataFrames – abordagem robusta que evita ambiguidades do leitor JSON nativo do Spark com arrays multi-linha.

In [ ]:
# --- Offers ---
offers_pdf = pd.read_json(os.path.join(RAW, "offers.json"))
offers_df = spark.createDataFrame(offers_pdf)
print(f"Offers: {offers_df.count()} linhas, {len(offers_df.columns)} colunas")
offers_df.printSchema()
offers_df.show(truncate=False)

In [ ]:
# --- Profile ---
profile_pdf = pd.read_json(os.path.join(RAW, "profile.json"))
profile_df = spark.createDataFrame(profile_pdf)
print(f"Profile: {profile_df.count()} linhas, {len(profile_df.columns)} colunas")
profile_df.printSchema()
profile_df.describe().show()

In [ ]:
# --- Transactions ---
# Quirk nos dados: o campo de ID da oferta varia por tipo de evento:
#   - "offer received" e "offer viewed"  → chave "offer id"  (com espaço)
#   - "offer completed"                  → chave "offer_id"  (com underscore)
# Usamos `or` para capturar o valor não-nulo de qualquer uma das chaves.
txn_pdf = pd.read_json(os.path.join(RAW, "transactions.json"))

txn_pdf["offer_id_raw"] = txn_pdf["value"].apply(
    lambda x: (x.get("offer id") or x.get("offer_id")) if isinstance(x, dict) else None
)
txn_pdf["amount"] = txn_pdf["value"].apply(
    lambda x: x.get("amount") if isinstance(x, dict) else None
)
txn_pdf["reward"] = txn_pdf["value"].apply(
    lambda x: x.get("reward") if isinstance(x, dict) else None
)
txn_pdf = txn_pdf.drop(columns=["value"])

transactions_df = spark.createDataFrame(txn_pdf)
print(f"Transactions: {transactions_df.count()} linhas, {len(transactions_df.columns)} colunas")
transactions_df.printSchema()
transactions_df.groupBy("event").count().orderBy(F.desc("count")).show()

# Verificar que offer_id_raw foi corretamente extraído para cada tipo de evento
print("Nulos em offer_id_raw por tipo de evento (esperado: nulo apenas em 'transaction'):")
transactions_df.groupBy("event").agg(
    F.count("*").alias("total"),
    F.count(F.when(F.col("offer_id_raw").isNull(), 1)).alias("offer_id_nulo")
).show()

## 3. Análise de Qualidade dos Dados

In [ ]:
print("=== Perfil de Nulos – Profile ===")
profile_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in profile_df.columns
]).show()

print("=== Distribuição de 'age' (top 5) ===")
profile_df.groupBy("age").count().orderBy(F.desc("count")).show(5)

print("=== Distribuição de 'gender' ===")
profile_df.groupBy("gender").count().orderBy(F.desc("count")).show()

In [ ]:
print("=== Distribuição de eventos ===")
transactions_df.groupBy("event").count().orderBy(F.desc("count")).show()

print("=== Range temporal ===")
transactions_df.agg(
    F.min("time_since_test_start").alias("min_t"),
    F.max("time_since_test_start").alias("max_t"),
    F.countDistinct("account_id").alias("clientes_unicos")
).show()

print("=== Valores ausentes em transactions ===")
transactions_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in transactions_df.columns
]).show()

## 4. Limpeza e Feature Engineering – Profile

In [ ]:
# Referência para cálculo de account_age: último dia do período do teste
REF_DATE = F.lit("2018-12-31").cast("date")

profile_clean = (
    profile_df
    # age=118 é sentinela para perfil incompleto (sem cadastro real)
    .withColumn("age", F.when(F.col("age") == 118, None).otherwise(F.col("age")))
    .withColumn("registered_on",
        F.to_date(F.col("registered_on").cast("string"), "yyyyMMdd"))
    .withColumn("account_age_days",
        F.datediff(REF_DATE, F.col("registered_on")))
    # Flag: perfil com dados demográficos completos
    .withColumn("is_complete_profile",
        F.when(
            F.col("age").isNotNull() & F.col("gender").isNotNull(), 1
        ).otherwise(0))
    .withColumn("gender", F.coalesce(F.col("gender"), F.lit("Unknown")))
    .withColumnRenamed("id", "account_id")
)

print(f"Profile limpo: {profile_clean.count()} clientes")
print(f"Perfis completos: {profile_clean.filter(F.col('is_complete_profile')==1).count()}")
print(f"Perfis incompletos: {profile_clean.filter(F.col('is_complete_profile')==0).count()}")
profile_clean.describe(["age", "credit_card_limit", "account_age_days"]).show()

## 5. Parsing dos Eventos de Transação

In [ ]:
# Separar eventos por tipo
events_received = (
    transactions_df
    .filter(F.col("event") == "offer received")
    .select(
        F.col("account_id"),
        F.col("offer_id_raw").alias("offer_id"),
        F.col("time_since_test_start").alias("received_at")
    )
)

events_viewed = (
    transactions_df
    .filter(F.col("event") == "offer viewed")
    .select(
        F.col("account_id"),
        F.col("offer_id_raw").alias("offer_id"),
        F.col("time_since_test_start").alias("viewed_at")
    )
)

events_completed = (
    transactions_df
    .filter(F.col("event") == "offer completed")
    .select(
        F.col("account_id"),
        F.col("offer_id_raw").alias("offer_id"),
        F.col("time_since_test_start").alias("completed_at"),
        F.col("reward")
    )
)

events_transaction = (
    transactions_df
    .filter(F.col("event") == "transaction")
    .select(
        F.col("account_id"),
        F.col("time_since_test_start"),
        F.col("amount")
    )
)

print("Eventos por tipo:")
print(f"  offer received:  {events_received.count():>7,}")
print(f"  offer viewed:    {events_viewed.count():>7,}")
print(f"  offer completed: {events_completed.count():>7,}")
print(f"  transaction:     {events_transaction.count():>7,}")

## 6. Construção da Tabela de Interações Oferta×Cliente

Para cada par `(account_id, offer_id)`, determinamos se houve conversão:
- **BOGO / discount**: `label = 1` se o evento `offer completed` existe para o par.
- **informational**: `label = 1` se o evento `offer viewed` existe (não há completion).

In [ ]:
# Agrega visualizações por par
viewed_agg = (
    events_viewed
    .groupBy("account_id", "offer_id")
    .agg(
        F.count("*").alias("n_views"),
        F.min("viewed_at").alias("first_viewed_at")
    )
)

# Agrega completions por par
completed_agg = (
    events_completed
    .groupBy("account_id", "offer_id")
    .agg(
        F.count("*").alias("n_completions"),
        F.sum("reward").alias("total_reward")
    )
)

# Junta received com viewed e completed
interactions = (
    events_received
    .join(viewed_agg, ["account_id", "offer_id"], "left")
    .join(completed_agg, ["account_id", "offer_id"], "left")
    .withColumn("viewed", F.when(F.col("n_views") > 0, 1).otherwise(0))
    .withColumn("completed", F.when(F.col("n_completions") > 0, 1).otherwise(0))
    # Incorpora tipo da oferta para definir label correto
    .join(
        offers_df.select(
            F.col("id").alias("offer_id"),
            F.col("offer_type")
        ),
        "offer_id", "left"
    )
    .withColumn("label",
        F.when(F.col("offer_type") == "informational", F.col("viewed"))
        .otherwise(F.col("completed"))
    )
)

print(f"Total de interações: {interactions.count():,}")
print(f"\nTaxa de conversão por tipo de oferta:")
interactions.groupBy("offer_type", "label").count().orderBy("offer_type", "label").show()

print("Taxa geral de conversão:")
interactions.agg(F.avg("label").alias("conversion_rate")).show()

## 7. Feature Engineering – Comportamento Transacional do Cliente

Computamos métricas de comportamento de compra histórico para cada cliente, que serão usadas como features no modelo.

In [ ]:
customer_behavior = (
    events_transaction
    .groupBy("account_id")
    .agg(
        F.count("*").alias("n_transactions"),
        F.sum("amount").alias("total_spent"),
        F.avg("amount").alias("avg_transaction"),
        F.stddev("amount").alias("std_transaction"),
        F.min("amount").alias("min_transaction"),
        F.max("amount").alias("max_transaction"),
        F.min("time_since_test_start").alias("first_txn_t"),
        F.max("time_since_test_start").alias("last_txn_t"),
    )
    .withColumn("observation_window",
        F.col("last_txn_t") - F.col("first_txn_t") + 1)
    .withColumn("avg_daily_spend",
        F.when(F.col("observation_window") > 0,
            F.col("total_spent") / F.col("observation_window")
        ).otherwise(0))
    .drop("first_txn_t", "last_txn_t")
)

print(f"Clientes com transações: {customer_behavior.count():,}")
customer_behavior.describe().show()

## 8. Feature Engineering – Ofertas

In [ ]:
offers_features = (
    offers_df
    .withColumn("n_channels", F.size(F.col("channels")))
    .withColumn("has_email",  F.array_contains(F.col("channels"), "email").cast("int"))
    .withColumn("has_mobile", F.array_contains(F.col("channels"), "mobile").cast("int"))
    .withColumn("has_social", F.array_contains(F.col("channels"), "social").cast("int"))
    .withColumn("has_web",    F.array_contains(F.col("channels"), "web").cast("int"))
    .withColumnRenamed("id", "offer_id")
    .drop("channels")
)

print("Ofertas com features expandidas:")
offers_features.show(truncate=False)

## 9. Dataset Unificado

Join de todas as dimensões na tabela de interações.

In [ ]:
final_df = (
    interactions
    # Dados do cliente
    .join(profile_clean, "account_id", "left")
    # Features da oferta (sem duplicar offer_type que já veio do join anterior)
    .join(
        offers_features.drop("offer_type"),
        "offer_id", "left"
    )
    # Comportamento transacional
    .join(customer_behavior, "account_id", "left")
    # Features de interação cliente×oferta
    .withColumn("discount_to_limit_ratio",
        F.when(F.col("credit_card_limit") > 0,
            F.col("discount_value") / F.col("credit_card_limit")
        ).otherwise(None))
    .withColumn("min_value_vs_avg_spend",
        F.when(F.col("avg_transaction") > 0,
            F.col("min_value") / F.col("avg_transaction")
        ).otherwise(None))
    # Remove colunas auxiliares de contagem
    .drop("n_views", "n_completions", "first_viewed_at", "total_reward",
          "received_at", "viewed", "completed")
)

print(f"Dataset final: {final_df.count():,} linhas × {len(final_df.columns)} colunas")
final_df.printSchema()

In [ ]:
print("=== Resumo do Dataset Final ===")
final_df.agg(
    F.count("*").alias("total_interacoes"),
    F.countDistinct("account_id").alias("clientes_unicos"),
    F.countDistinct("offer_id").alias("ofertas_unicas"),
    F.avg("label").alias("taxa_conversao_geral"),
).show()

print("Taxa de conversão por tipo de oferta:")
final_df.groupBy("offer_type") \
    .agg(
        F.count("*").alias("n"),
        F.avg("label").alias("conversion_rate"),
        F.sum("label").alias("n_converted")
    ).orderBy("offer_type").show()

print("Nulos no dataset final:")
null_counts = final_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in final_df.columns
])
null_counts.show()

## 10. Salvando os Dados Processados

In [ ]:
# Salva em Parquet (eficiente para leitura distribuída)
parquet_path = os.path.join(PROCESSED, "interactions_features.parquet")
final_df.write.mode("overwrite").parquet(parquet_path)
print(f"Parquet salvo em: {parquet_path}")

# Salva em CSV para uso no notebook de modelagem (sem Spark)
csv_path = os.path.join(PROCESSED, "interactions_features.csv")
final_df.toPandas().to_csv(csv_path, index=False)
print(f"CSV salvo em: {csv_path}")

print("\nProcessamento concluído com sucesso!")